# Medical Triage Agent AI POC
## Google Colab Training Pipeline


In [ ]:
from __future__ import annotations
import logging
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger('training_pipeline')


# Section 2 — Bootstrap Runtime


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
def ensure_package(module_name,pip_name=None):
    if importlib.util.find_spec(module_name) is not None:
        return
    subprocess.run([sys.executable,'-m','pip','install','-U',pip_name or module_name],check=True)


In [ ]:
DEPENDENCIES=[('torch',None),('transformers',None),('datasets',None),('accelerate',None),('trl',None),('peft',None),('bitsandbytes',None),('huggingface_hub','huggingface_hub'),('wandb',None),('evaluate',None),('sentencepiece',None),('safetensors',None)]
for m,p in DEPENDENCIES:
    ensure_package(m,p)


In [ ]:
PROJECT_ROOT=Path.cwd().resolve()
while PROJECT_ROOT!=PROJECT_ROOT.parent:
    if (PROJECT_ROOT/'backend').exists() and (PROJECT_ROOT/'frontend').exists():
        break
    PROJECT_ROOT=PROJECT_ROOT.parent
else:
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))


In [ ]:
required=['backend/app/training/colab','backend/app/training/sft','backend/app/training/dpo','backend/app/training/evaluation']
missing=[p for p in required if not (PROJECT_ROOT/Path(p)).exists()]
if missing:
    raise FileNotFoundError(missing)


# Section 3


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
from backend.app.training.colab.colab_environment import log_environment_info
from backend.app.training.colab.colab_gpu_detector import log_gpu_info
from backend.app.training.colab.colab_hf_login import ensure_hf_login
from backend.app.training.colab.colab_checkpoint_sync import create_default_checkpoint_sync
environment=log_environment_info()
gpu_info=log_gpu_info()
hf_status=ensure_hf_login()
checkpoint_sync=create_default_checkpoint_sync(output_dir='./outputs')
checkpoint_sync.get_status()


# Section 4


In [ ]:
import runpy
runpy.run_module('backend.app.training.sft.train_sft',run_name='__main__')


# Section 5


In [ ]:
import runpy
runpy.run_module('backend.app.training.dpo.train_dpo',run_name='__main__')


# Section 6


In [ ]:
import runpy
runpy.run_module('backend.app.training.evaluation.clinical_eval_runner',run_name='__main__')


# Section 7


In [ ]:
from backend.app.training.colab.colab_checkpoint_sync import create_default_checkpoint_sync
checkpoint_sync=create_default_checkpoint_sync(output_dir='./outputs')
print(checkpoint_sync.sync_latest_checkpoint())


# Section 8


In [ ]:
from backend.app.training.colab.colab_checkpoint_sync import create_default_checkpoint_sync
checkpoint_sync=create_default_checkpoint_sync(output_dir='./outputs')
print(checkpoint_sync.get_status())
